# FDSI benchmark v1.1: fixed (no-adaptation) baseline from the calibration end

v1.0's `../fdsi_benchmark/02_fixed_adaptation.ipynb` ran the calibrated model, with every
adaptation flag off, over the whole recording with `process_data`. Here the same model and
config run with `process_from_calib_end`: CBSS's own output over the calibration window
[0, 5 s), the unadapted model from 5 s onwards. Every v1.1 adapted config is compared with this
baseline, so a v1.1 gain is attributable to adaptation rather than to where adaptation starts.

Computes whatever is missing under `OUTPUTS_ROOT/adaptation_v1_1/<sub>/fixed/`
(`RUN_DECOMP_FIXED=True`); a rerun only loads.

## Config

In [ ]:
RUN_DECOMP_FIXED = True   # compute missing adaptation_v1_1/<sub>/fixed/..._adapt_fixed.pkl

import sys
from pathlib import Path
sys.path.insert(0, '../..')          # package root
sys.path.insert(0, str(Path.cwd()))  # this folder, for fdsi_v11 (which imports ../fdsi_benchmark/fdsi_common)

import fdsi_v11 as fv
fc = fv.fc

g = fv.Grid.from_yaml(Path('../../configs/data_configs/fdsi_benchmark_grid.yaml'))
BASE_CONFIG_PATH = Path('../../configs/adapt_configs/default_muniverse.yaml')   # v1.0's base config

print(f'{len(g.recordings())} recordings; calibration window [0, {g.cal_end}) samples; '
      f'pool {g.pool_conditions} ({g.optim_sub}, {g.optim_snr} dB); holdout {g.holdout_conditions}')

## Calibration

The cached v1.0 calibrations (`../fdsi_benchmark/01_calibration.ipynb`), unchanged: both versions
track exactly the same units, so their results can be paired unit by unit.

In [ ]:
calibrations = fv.load_calibrations(g)
print(f'Calibration available: {sum(c is not None for c in calibrations.values())}/{len(calibrations)} recordings.')

## Apply the no-adaptation baseline to the full benchmark grid

`fv.fixed_config` is v1.0's baseline config (`AdaptConfig(ext_fact, adapt_wh=adapt_sv=adapt_sd=False)`),
on CPU; `fv.apply_from_calib_end` attaches SIL and RoA and caches each result exactly like v1.0's
`run_fixed`.

In [ ]:
fixed_results = fv.apply_all(g, calibrations, fv.fixed_config(g), None, RUN_DECOMP_FIXED)

## Example recording -- v1.0 vs v1.1 baseline

Outside the calibration window both run the same unadapted model, so they differ only through
batch alignment (v1.1's batches start at the window's end) and per-batch centring. Inside the
window v1.1 returns CBSS's own spikes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from adapt_decomp import AdaptationResult
from adapt_decomp.utils import load_gt
from adapt_decomp.utils.plots import plot_spikes

sub, cond, snr = g.optim_sub, g.triangular[0], g.optim_snr
cbss_result = calibrations[(sub, cond, snr)]
outputs = {
    'fixed (v1.1)': fixed_results[(sub, cond, snr)],
    'fixed (v1.0)': AdaptationResult.load(fc.fixed_paths(g.adapt_dir_v10, sub, cond, snr)[0]),
}

title = 'No adaptation execution time (ms per batch)'
print(title + '\n' + '-' * len(title))
for key in ['preprocess_time_ms', 'wh_time_ms', 'sv_time_ms', 'sd_time_ms', 'total_time_ms']:
    print(f'{key:>19}: ' + '   '.join(f'{label} {out[key].mean():.3f} ± {out[key].std():.3f}'
                                       for label, out in outputs.items()))

In [ ]:
n_samples = outputs['fixed (v1.1)'].spikes.shape[0]
gt_full_bin = load_gt(fc.gt_spikes_path(g.data_dir, sub, cond), n_samples=n_samples)
ts_full = np.arange(n_samples) / g.fs

ax = plot_spikes(
    spikes={**{label: out.spikes.numpy() for label, out in outputs.items()},
            'Ground truth': gt_full_bin[:, cbss_result.gt_matched_indices]},
    timestamps=ts_full,
    time_range=(0, ts_full[-1]),
    roa={label: out.roa for label, out in outputs.items()},
    sil={label: out.sil for label, out in outputs.items()},
)
ax.axvline(g.cal_end / g.fs, color='k', ls='--', lw=1)
ax.set_title(f'Output spikes vs ground truth -- {sub} {cond} snr{snr}dB (no adaptation; dashed: calibration end)')
plt.show()

## v1.0 vs v1.1 baseline over the grid

Paired per unit (same calibrations, same units): full recording, then after the calibration
window only. A non-zero difference here is the part of any later v1.0 vs v1.1 comparison that
comes from where adaptation starts rather than from adaptation itself.

In [ ]:
df_roa = fv.roa_table(g, fv.configs())
df_post = fv.window_roa_table(g, fv.configs(), slice(g.cal_end, None))
import pandas as pd
pd.concat({'full recording': fv.paired_summary(df_roa, fv.pairs()),
           'after calibration': fv.paired_summary(df_post, fv.pairs())}).round(3)